## Deep Research With Evaluator

Evaluates for satisfactory results, if not, it augments the search

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

In [3]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [5]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [7]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("ronaldkica@outlook.com")
    to_email = To("okello.ronald045@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [9]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)



In [10]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

In [ ]:
# Evaluator Agent: assesses search quality and suggests additional searches if gaps exist

class EvaluationResult(BaseModel):
    is_satisfactory: bool = Field(
        description="Whether the search results adequately cover the query for writing a comprehensive report."
    )
    missing_aspects: list[str] = Field(
        description="Key topics or gaps not covered by current results. Empty list if satisfactory."
    )
    additional_search_queries: list[str] = Field(
        description="Specific search terms to fill identified gaps. Empty list if satisfactory. Max 3 queries."
    )
    reasoning: str = Field(description="Brief explanation of your assessment.")


EVALUATOR_INSTRUCTIONS = """You are a research quality evaluator. You assess whether web search results 
adequately cover an original research query. Your job is to determine if a writer could produce a 
comprehensive, well-rounded report from these results alone.
If the results are thin, redundant, miss key angles, or lack recent/authoritative sources, mark as 
NOT satisfactory and provide 1-3 specific search queries to fill the gaps. Be concrete with search terms.
If the results cover the main aspects of the query with reasonable depth, mark as satisfactory."""

evaluator_agent = Agent(
    name="EvaluatorAgent",
    instructions=EVALUATOR_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=EvaluationResult,
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [12]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

In [ ]:
MAX_EVALUATION_ITERATIONS = 2


async def perform_additional_searches(queries: list[str]) -> list[str]:
    """Run search_agent for each query string to fill gaps identified by evaluator."""
    items = [WebSearchItem(reason="Filling identified research gap", query=q) for q in queries]
    plan = WebSearchPlan(searches=items)
    return await perform_searches(plan)


async def evaluate_and_augment(query: str, search_results: list[str]) -> list[str]:
    """Evaluate search results. If not satisfactory, run additional searches and re-evaluate (up to MAX_EVALUATION_ITERATIONS)."""
    all_results = list(search_results)

    for iteration in range(MAX_EVALUATION_ITERATIONS):
        print(f"Evaluating search results (iteration {iteration + 1})...")
        input_text = (
            f"Original query: {query}\n\n"
            f"Search results so far:\n"
            + "\n\n---\n\n".join(all_results)
        )
        result = await Runner.run(evaluator_agent, input_text)
        evaluation = result.final_output

        if evaluation.is_satisfactory:
            print("Search results satisfactory. Proceeding to writer.")
            return all_results

        if not evaluation.additional_search_queries:
            print("Not satisfactory but no additional queries suggested. Proceeding anyway.")
            return all_results

        print(f"Gaps identified: {evaluation.missing_aspects}")
        print(f"Performing {len(evaluation.additional_search_queries)} additional searches...")
        additional = await perform_additional_searches(evaluation.additional_search_queries)
        all_results.extend(additional)

    print("Max iterations reached. Proceeding with current results.")
    return all_results

### Pipeline flow with Evaluator

1. **Planner** → plans searches  
2. **Search** → performs initial searches  
3. **Evaluator** → assesses quality; if gaps exist, runs additional searches and re-evaluates (max 2 iterations)  
4. **Writer** → produces report from final results  
5. **Email** → sends report

### The next 2 functions write a report and email it

In [14]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Showtime!

In [ ]:
query = "Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    search_results = await evaluate_and_augment(query, search_results)
    report = await write_report(query, search_results)
    await send_email(report)
    print("Hooray!")


